# 🧠 FAC-Synthesis: Phase 4b — SAE Scoring & Contrastive Pairs

Αυτό το notebook εκτελεί το **Δεύτερο Μισό** της παραγωγής δεδομένων.

Αφού το Abliterated μοντέλο έγραψε τα τοξικά queries στο `step1_queries.queries.tsv` (Phase 4a),
τώρα χρησιμοποιούμε το **Official Censored `meta-llama/Llama-3.1-8B-Instruct`** (4-bit quantized)
μαζί με το SAE (`Zhongzhi1228/sae_llama_l16_h65536`, layer 16, 65k features) για να μετρήσουμε
ποια queries πετυχαίνουν τα μεγαλύτερα activations.

### ⚠️ ΠΡΙΝ ΞΕΚΙΝΗΣΕΙΣ
1. Πρέπει να έχεις **ήδη τρέξει** το `phase4a_synthesis.ipynb`
2. Κάνε **Runtime → Disconnect and delete runtime** (για να αδειάσει η GPU)
3. Σιγουρέψου ότι έχεις **L4 ή T4 GPU** (δες παρακάτω)

### 🔐 Hugging Face Access Token
1. Φτιάξε λογαριασμό στο [Hugging Face](https://huggingface.co/).
2. Πήγαινε στη σελίδα του [Llama-3.1-8B-Instruct](https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct) και πάτα αποδοχή των όρων χρήσης.
3. Πήγαινε στα [Settings > Access Tokens](https://huggingface.co/settings/tokens) και φτιάξε ένα νέο Token (τύπου Read).
4. Στο μενού αριστερά στο Colab, πάτα το εικονίδιο 🔑 (Secrets) και πρόσθεσε ένα secret με όνομα `HF_TOKEN` και τιμή το token σου.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Επιβεβαίωση ότι έχουμε GPU (T4 ή L4)
!nvidia-smi

## 1. Εγκατάσταση Βιβλιοθηκών

In [ ]:
!pip install -q transformers==4.43.4 accelerate==0.33.0 bitsandbytes datasets

## 2. Σύνδεση με Hugging Face

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Θα τραβήξει αυτόματα το κλειδί που έβαλες στα Secrets του Colab
hf_token = userdata.get("HF_TOKEN")
login(hf_token)

## 3. Λήψη Κώδικα (FAC-Synthesis)

In [ ]:
%%bash
git clone https://github.com/michalispsy/SLP_2026_SEMESTER_EXER.git FAC-Synthesis

## 4. Patch: 4-bit Loading για `generator.py`
Το Llama 3.1 8B κανονικά απαιτεί 16GB VRAM. Πατσάρουμε το `generator.py` (που χρησιμοποιεί το `collect_spans.py`) για να φορτώσει σε 4-bit (~6GB VRAM), **ακριβώς όπως στο cleaned.ipynb**.

In [ ]:
import os

path = "/content/FAC-Synthesis/sae_feature_analysis/interpret_features/generator.py"
with open(path) as f:
    src = f.read()

# (a) Fix CACHE_DIR placeholder
src = src.replace(
    'CACHE_DIR = "xxx/.cache/huggingface"',
    'CACHE_DIR = os.environ.get("HF_CACHE_DIR", "/root/.cache/huggingface")'
)

# (b) 4-bit quantization patch (ίδιο με cleaned.ipynb)
old_load = '''        self._model = trf.AutoModelForCausalLM.from_pretrained(
            self._name,
            cache_dir=CACHE_DIR,
            torch_dtype=self._dtype,
            device_map=maps
        )'''

new_load = '''        from transformers import BitsAndBytesConfig
        _use_4bit = os.environ.get("FAC_USE_4BIT", "1") == "1"
        if _use_4bit and self._device != "cpu":
            _bnb = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=tc.float16,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
            )
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                quantization_config=_bnb, device_map=maps)
        else:
            self._model = trf.AutoModelForCausalLM.from_pretrained(
                self._name, cache_dir=CACHE_DIR,
                torch_dtype=self._dtype, device_map=maps)'''

src = src.replace(old_load, new_load)

with open(path, 'w') as f:
    f.write(src)
print('✅ Patched generator.py (CACHE_DIR + 4-bit quantization)')

## 5. Κατέβασμα SAE Checkpoint
Κατεβάζουμε δυναμικά το SAE checkpoint από το HF repo (ίδιο setup με cleaned.ipynb, Cell 13+15).
Αν το filename δεν ταιριάζει με `{cls}_l{layer}_*.pth`, κάνουμε symlink σε σωστό όνομα.

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download
import os, shutil

SAE_REPO = 'Zhongzhi1228/sae_llama_l16_h65536'

files = list_repo_files(SAE_REPO)
print('Files στο repo:')
for f in files:
    print(' -', f)

# Βρες δυναμικά το .pth αρχείο
pth_files = [f for f in files if f.endswith('.pth')]
assert pth_files, 'Δεν βρέθηκε .pth αρχείο στο repo!'
SAE_FILENAME = pth_files[0]
print(f'\nΘα κατεβάσω: {SAE_FILENAME}')

SAE_PATH = hf_hub_download(repo_id=SAE_REPO, filename=SAE_FILENAME)
print(f'\n✅ SAE downloaded: {SAE_PATH}')

In [ ]:
# Αν το filename δεν ταιριάζει με {cls}_l{layer}_*.pth, symlink σε σωστό όνομα
# (ίδιος κώδικας με cleaned.ipynb Cell 15)
import os, shutil

basename = os.path.basename(SAE_PATH)
if not (basename.startswith(('topk_l', 'sae_l', 'ae_l', 'topk5_l', 'topk6_l', 'topk7_l'))):
    target_name = 'topk_l16_h65536.pth'
    target_path = f'/content/{target_name}'
    if not os.path.exists(target_path):
        shutil.copy(SAE_PATH, target_path)
    SAE_PATH = target_path
    print(f'⚠️ Μετονομάστηκε σε: {SAE_PATH}')
else:
    print(f'✅ Το όνομα ταιριάζει: {basename}')

print(f'\nΤελικό SAE_PATH: {SAE_PATH}')

## 6. Αντιγραφή Missing Features
Χρειαζόμαστε και εδώ το TSV με τα 318 features για τα analyze/merge scripts.

In [ ]:
%%bash
# Αντιγραφή από το repo (ή από το Drive αν το έχεις εκεί)
cp "/content/FAC-Synthesis/our_work/synthesis/synthesis_data/steps_4a,b,c/intersection_tox7_corr3___MISS_FEATURES__INPUT_FILE_STEP_4A_FIXED.tsv" /content/missing_features.tsv

# Επιβεβαίωση
echo "Γραμμές (features + header):"
wc -l /content/missing_features.tsv
echo "\nΠρώτες 3 γραμμές:"
head -3 /content/missing_features.tsv

## 7. Επιβεβαίωση: Υπάρχουν τα αρχεία του Phase 4a στο Drive;
Πριν τρέξουμε οτιδήποτε, ελέγχουμε ότι τα queries του Phase 4a υπάρχουν στο Drive.

In [ ]:
import os

queries_path = "/content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/step1_queries.queries.tsv"

if os.path.exists(queries_path):
    lines = sum(1 for _ in open(queries_path))
    print(f"✅ Βρέθηκε! ({lines} γραμμές)")
    print(f"📁 {queries_path}")
else:
    print("❌ ΔΕΝ βρέθηκε! Σιγουρέψου ότι έτρεξες πρώτα το Phase 4a.")
    print(f"   Αναζήτηση: {queries_path}")

## 8. Collect Spans — SAE Scoring
Περνάμε τα τοξικά queries μέσα από το **Official Censored Llama 3.1** + SAE.

Το script:
1. Φορτώνει αυτόματα το `meta-llama/Llama-3.1-8B-Instruct` (4-bit)
2. Κουμπώνει πάνω του το SAE (layer 16)
3. Διαβάζει κάθε query, κάνει forward pass, μετράει activations
4. Γράφει τα αποτελέσματα **ζωντανά στο Drive**

**⏱ Εκτίμηση χρόνου:** ~20-40 λεπτά για 636 queries σε L4.

In [ ]:
import os

# Δημιουργία output φακέλου στο Drive (live save!)
os.makedirs('/content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out', exist_ok=True)

# Περνάμε το SAE_PATH σαν env variable για να το δει το ! command
os.environ['SAE_PATH_4B'] = SAE_PATH

# Χρήση ! αντί %%bash για live output
! cd /content/FAC-Synthesis/sae_feature_analysis/interpret_features/ && python collect_spans.py 0 llama 0 1 \
    --data-path /content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/step1_queries.queries.tsv \
    --threshold 0.0 \
    --sae-path $SAE_PATH_4B \
    --out-dir /content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out

## 9. Sanity Check: Inspect Activations

In [ ]:
import pandas as pd

spans_path = "/content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0/textspans_group0.tsv"
df = pd.read_csv(spans_path, sep="\t")

print(f"Total activation rows: {len(df)}")
print(f"Unique NeuronIDs: {df['NeuronID'].nunique()}")
print(f"Score range: {df['Score'].min():.4f} — {df['Score'].max():.4f}")
df.head(20)

## 10. Groupby TextSpans
Ομαδοποίηση: το `collect_spans` γράφει `textspans_group0.tsv`, το `groupby_textspans.py` περιμένει `full.tsv`.

In [ ]:
import shutil

folder_synth = "/content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0"
shutil.copy(f"{folder_synth}/textspans_group0.tsv", f"{folder_synth}/full.tsv")
print("✅ Copied to full.tsv")

! cd /content/FAC-Synthesis/sae_feature_analysis/interpret_features && python groupby_textspans.py "{folder_synth}"

## 11. Analyze Step 1 Synthetic Data
Βρίσκει τα Top-2 queries με το υψηλότερο activation score ανά feature.

In [ ]:
! cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python analyze_step1_synthetic_data.py \
  --final-decision-file /content/missing_features.tsv \
  --textspans-file /content/drive/MyDrive/fac_synthesis/step_4/4b_synthetic_out/threshold_0.0/textspans_group0.tsv \
  --synthetic-queries-file /content/drive/MyDrive/fac_synthesis/step_4/4a/log_files/step1_queries.queries.tsv \
  --output-jsonl /content/drive/MyDrive/fac_synthesis/step_4/4b_step1_analyzed.jsonl

## 12. Merge & Create Contrastive Pairs
Δημιουργία του τελικού αρχείου `step1_contrastive_pairs.jsonl` που πάει στο Phase 4c (Round 2).

In [ ]:
! cd /content/FAC-Synthesis/fac_synthesis/step1_contrastive_pair_construction/ && python merge_step1_failed_cases.py \
  --final-decision-file /content/missing_features.tsv \
  --triplets-file /content/drive/MyDrive/fac_synthesis/step_4/4b_step1_analyzed.jsonl \
  --output-file /content/drive/MyDrive/fac_synthesis/step_4/step1_contrastive_pairs.jsonl

## 13. Επιβεβαίωση Τελικού Αποτελέσματος

In [ ]:
import json

pairs_path = "/content/drive/MyDrive/fac_synthesis/step_4/step1_contrastive_pairs.jsonl"

with open(pairs_path) as f:
    records = [json.loads(line) for line in f]

print(f"📊 Total contrastive pairs: {len(records)}")
print(f"🔑 Keys per record: {list(records[0].keys()) if records else 'N/A'}")
print(f"\n📋 First 3 records:")
for r in records[:3]:
    print(json.dumps(r, ensure_ascii=False, indent=2)[:500])
    print('---')

print(f"\n✅ ΤΕΛΟΣ! Το step1_contrastive_pairs.jsonl είναι στο Drive!")
print(f"📁 {pairs_path}")